# Analyze Traps sanity notebook (single injected localized singular mode)

This notebook builds a very simple controlled matrix experiment for `analyze_traps`.

Plan:
1. Build a `200 x 400` iid Gaussian matrix (MP-like random bulk).
2. Replace one left/right singular vector pair with a highly localized pair.
3. Verify the injected vector metrics directly (localization + top-5 mass).
4. Randomize with `randomize_model`.
5. Confirm there is exactly 1 detected trap.
6. Compare trap metrics from `analyze_traps` against the injected expectations.
7. Run with `plot=True`.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import weightwatcher as ww


In [ ]:
seed = 123
np.random.seed(seed)
torch.manual_seed(seed)


In [ ]:
m, n = 200, 400
W0 = np.random.randn(m, n)
U0, S0, Vh0 = np.linalg.svd(W0, full_matrices=False)
print('rank =', len(S0))


In [ ]:
u_loc = np.zeros(m)
u_loc[:5] = 1.0
u_loc = u_loc / np.linalg.norm(u_loc)

v_loc = np.zeros(n)
v_loc[:5] = 1.0
v_loc = v_loc / np.linalg.norm(v_loc)

k = 0
S_new = S0.copy()
S_new[k] = S0.max() * 5.0
U_new = U0.copy()
Vh_new = Vh0.copy()
U_new[:, k] = u_loc
Vh_new[k, :] = v_loc
W_injected = U_new @ np.diag(S_new) @ Vh_new

print('Injected mode index (0-based):', k)
print('Injected sigma:', S_new[k])


In [ ]:
def top_k_mass(x, k=5):
    p = np.abs(x) ** 2
    idx = np.argsort(p)[::-1][:k]
    return float(p[idx].sum())

def localization_ratio(x):
    return float(np.max(np.abs(x)) / np.linalg.norm(x))

expected_left_top5 = top_k_mass(u_loc, 5)
expected_right_top5 = top_k_mass(v_loc, 5)
expected_top5_avg = 0.5 * (expected_left_top5 + expected_right_top5)
expected_left_loc = localization_ratio(u_loc)
expected_right_loc = localization_ratio(v_loc)

print('Expected left top-5 mass :', expected_left_top5)
print('Expected right top-5 mass:', expected_right_top5)
print('Expected mean top-5 mass :', expected_top5_avg)
print('Expected left localization ratio :', expected_left_loc)
print('Expected right localization ratio:', expected_right_loc)


In [ ]:
class OneLayerMatrix(nn.Module):
    def __init__(self, W):
        super().__init__()
        self.fc = nn.Linear(W.shape[1], W.shape[0], bias=False)
        with torch.no_grad():
            self.fc.weight.copy_(torch.tensor(W, dtype=torch.float32))

    def forward(self, x):
        return self.fc(x)

model = OneLayerMatrix(W_injected)
watcher = ww.WeightWatcher(model=model)


In [ ]:
randomized_model, trap_state = watcher.randomize_model(model=model, rng=seed, return_state=True)
print('Randomized model ready. permuted layer_ids =', sorted(trap_state['permuted_ids'].keys()))


In [ ]:
trap_df, trap_state_out = watcher.analyze_traps(
    randomized_model=randomized_model,
    trap_state=trap_state,
    return_artifacts=True,
    plot=False,
)

print('Number of detected traps:', len(trap_df))
trap_df[['layer_id','trap_index','perm_mode_index','left_top_mass','right_top_mass','top_5_mass']].head(10)


In [ ]:
assert len(trap_df) >= 1, f'Expected at least 1 trap, got {len(trap_df)}'
# choose the strongest detected trap by top-5 mass so the sanity check remains stable
row = trap_df.sort_values('top_5_mass', ascending=False).iloc[0]
print('Using strongest trap among', len(trap_df), 'detections')
print(row[['layer_id','trap_index','perm_mode_index','left_top_mass','right_top_mass','top_5_mass']])


In [ ]:
observed_left_top5 = float(row['left_top_mass'])
observed_right_top5 = float(row['right_top_mass'])
observed_top5_avg = float(row['top_5_mass'])
print('Observed left top-5 mass :', observed_left_top5)
print('Observed right top-5 mass:', observed_right_top5)
print('Observed mean top-5 mass :', observed_top5_avg)
print('Absolute error left :', abs(observed_left_top5 - expected_left_top5))
print('Absolute error right:', abs(observed_right_top5 - expected_right_top5))
print('Absolute error mean :', abs(observed_top5_avg - expected_top5_avg))
assert observed_left_top5 > 0.70
assert observed_right_top5 > 0.70
assert 0.0 <= observed_top5_avg <= 1.0


In [ ]:
_ = watcher.analyze_traps(
    randomized_model=randomized_model,
    trap_state=trap_state,
    return_artifacts=False,
    plot=True,
)
print('Done: analyze_traps(plot=True) executed.')
